<a href="https://colab.research.google.com/github/ajinfajrian/data-science-2026/blob/master/Pertemuan6_Fajrian_Ichlasul_240401020100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SEL 1: Memuat Dataset & Analisis Missing Value Awal

Instruksi Modul: Memuat dataset Titanic kustom, memfilter kolom sesuai target pemodelan, dan mengecek sisa data yang kosong sebelum proses pembagian feature-target.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Memuat dataset Titanic eksternal resmi dari Seaborn
df_raw = sns.load_dataset('titanic')

# 2. Memilih kolom spesifik sesuai instruksi modul pertemuan 6
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df_raw[cols].copy()

print("--- Informasi Awal Dataset ---")
print(f"Shape dataset: {df.shape}")
print("\nMissing values per kolom:")
print(df.isnull().sum())

print("\n--- Distribusi Kelas Target (Survived) ---")
print(df['survived'].value_counts(normalize=True).round(3))

--- Informasi Awal Dataset ---
Shape dataset: (891, 8)

Missing values per kolom:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

--- Distribusi Kelas Target (Survived) ---
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


SEL 2: Penanganan Imputasi Cepat & Pembagian Variabel (X dan y)

Instruksi Modul: Mengisi kolom numerik yang masih bolong (age) dengan median dan kolom kategorikal (embarked) dengan modus agar siap diproses oleh library Scikit-Learn, lalu memisahkan Fitur (X) dan Target (y).

In [2]:
# 1. Imputasi data bolong agar aman saat scaling/encoding
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# 2. Memisahkan Fitur/Independen (X) dan Target/Dependen (y)
X = df.drop('survived', axis=1)
y = df['survived']

print("Variabel X (Fitur) dan y (Target) berhasil dipisahkan.")
print(f"Dimensi X: {X.shape} | Dimensi y: {y.shape}")

Variabel X (Fitur) dan y (Target) berhasil dipisahkan.
Dimensi X: (891, 7) | Dimensi y: (891,)


SEL 3: Pembagian Data Train & Test (Train-Test Split dengan Stratify)

Instruksi Modul: Membagi dataset menjadi 80% untuk data Train (pelatihan) dan 20% untuk data Test (pengujian). Wajib menggunakan parameter stratify=y agar proporsi kelas target tetap seimbang di kedua bagian data dan menghindari data leakage.

In [3]:
# Melakukan data split dengan rasio 80:20 dan penguncian acak random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Jumlah baris X_train (Data Latih): {X_train.shape[0]}")
print(f"Jumlah baris X_test (Data Uji)  : {X_test.shape[0]}")

print("\nProporsi target 'survived' di Data Latih:")
print(y_train.value_counts(normalize=True).round(3))

print("\nProporsi target 'survived' di Data Uji:")
print(y_test.value_counts(normalize=True).round(3))

Jumlah baris X_train (Data Latih): 712
Jumlah baris X_test (Data Uji)  : 179

Proporsi target 'survived' di Data Latih:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi target 'survived' di Data Uji:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


SEL 4: Mengonversi Data Kategorikal (Categorical Encoding via One-Hot)

Instruksi Modul: Mengubah kolom teks kategorikal non-numerik (sex dan embarked) menjadi angka biner 0 dan 1 menggunakan metode One-Hot Encoding agar mesin bisa membacanya tanpa mengasumsikan adanya tingkatan derajat nilai.

In [4]:
# 1. Menentukan kolom kategori yang akan di-encode
kategori_cols = ['sex', 'embarked']

# 2. Inisialisasi OneHotEncoder (handle_unknown='ignore' untuk mengantisipasi data baru di masa depan)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 3. Fit dan Transform pada data Train, dan hanya Transform pada data Test (Mencegah Data Leakage!)
X_train_encoded = encoder.fit_transform(X_train[kategori_cols])
X_test_encoded = encoder.transform(X_test[kategori_cols])

# 4. Membuat DataFrame baru hasil encoding dengan nama fitur yang sesuai
encoded_cols_names = encoder.get_feature_names_out(kategori_cols)
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoded_cols_names, index=X_train.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoded_cols_names, index=X_test.index)

print("--- Hasil Nama Kolom Baru Setelah One-Hot Encoding ---")
print(encoded_cols_names)

--- Hasil Nama Kolom Baru Setelah One-Hot Encoding ---
['sex_female' 'sex_male' 'embarked_C' 'embarked_Q' 'embarked_S']


SEL 5: Standardisasi Fitur Numerik (Feature Scaling via StandardScaler)

Instruksi Modul: Menyamakan skala nilai kolom numerik kontinu (age dan fare) menggunakan StandardScaler (Z-score Scaling) agar rentang nilainya berpusat pada nilai rata-rata (mean) = 0 dan standar deviasi = 1.

In [5]:
# 1. Menentukan kolom numerik yang perlu disamakan skalanya
numerik_cols = ['age', 'fare', 'sibsp', 'parch']

# 2. Inisialisasi StandardScaler
scaler = StandardScaler()

# 3. Fit & Transform pada data Train, dan hanya Transform pada data Test
X_train_scaled = scaler.fit_transform(X_train[numerik_cols])
X_test_scaled = scaler.transform(X_test[numerik_cols])

# 4. Membuat DataFrame baru hasil scaling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=numerik_cols, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=numerik_cols, index=X_test.index)

print("Proses standardisasi fitur numerik selesai.")
print("Sampel nilai rata-rata data Train setelah scaling:", np.mean(X_train_scaled, axis=0).round(2))

Proses standardisasi fitur numerik selesai.
Sampel nilai rata-rata data Train setelah scaling: [ 0. -0. -0. -0.]


SEL 6: Menggabungkan Seluruh Fitur (Final Preprocessed Data)

Instruksi Modul: Menggabungkan kembali fitur numerik yang sudah diskalakan, fitur kategorikal yang sudah di-encode, serta fitur ordinal bawaan (pclass) menjadi satu kesatuan DataFrame final yang siap dimasukkan ke algoritma Machine Learning di pertemuan 7.

In [6]:
# 1. Menggabungkan kolom pclass (yang tidak di-scale/encode) dengan dataframe numerik & kategori
X_train_final = pd.concat([X_train[['pclass']], X_train_scaled_df, X_train_encoded_df], axis=1)
X_test_final = pd.concat([X_test[['pclass']], X_test_scaled_df, X_test_encoded_df], axis=1)

print("--- 5 Baris Pertama Data Train Siap Pakai ---")
print(X_train_final.head())
print(f"\nUkuran Akhir Fitur Latih: {X_train_final.shape}")
print(f"Ukuran Akhir Fitur Uji  : {X_test_final.shape}")

--- 5 Baris Pertama Data Train Siap Pakai ---
     pclass       age      fare     sibsp     parch  sex_female  sex_male  \
692       3 -0.112078  0.513812 -0.465084 -0.466183         0.0       1.0   
481       2 -0.112078 -0.662563 -0.465084 -0.466183         0.0       1.0   
527       1 -0.112078  3.955399 -0.465084 -0.466183         0.0       1.0   
855       3 -0.879807 -0.467874 -0.465084  0.727782         1.0       0.0   
801       2  0.118241 -0.115977  0.478335  0.727782         1.0       0.0   

     embarked_C  embarked_Q  embarked_S  
692         0.0         0.0         1.0  
481         0.0         0.0         1.0  
527         0.0         0.0         1.0  
855         0.0         0.0         1.0  
801         0.0         0.0         1.0  

Ukuran Akhir Fitur Latih: (712, 10)
Ukuran Akhir Fitur Uji  : (179, 10)
